### 2.1

输入：$3 \times 32 \times 32$，卷积核 $16$ 个，大小 $3 \times 5 \times 5$，Padding=2，Stride=2。

**1. 输出特征图尺寸**

$$
H_{out} = \left\lfloor \frac{H_{in} + 2p - k}{s} \right\rfloor + 1
= \left\lfloor \frac{32 + 4 - 5}{2} \right\rfloor + 1
= \lfloor 15.5 \rfloor + 1 = 16
$$

同理 $W_{out}=16$，通道数等于卷积核个数 $=16$，输出尺寸为 $\mathbf{16 \times 16 \times 16}$。

**2. 单个输出像素所需乘法次数**

单个卷积核大小为 $3 \times 5 \times 5$：

$$3 \times 5 \times 5 = \mathbf{75} \text{ 次}$$

In [1]:
# 2.2
import numpy as np
import torch
import torch.nn.functional as F

def max_pool2d(x, kernel_size, stride=1, padding=0):
    C, H, W = x.shape
    kH = kW = kernel_size
    sH = sW = stride
    pH = pW = padding
    x_pad = np.pad(x, ((0,0),(pH,pH),(pW,pW)), mode='constant', constant_values=-np.inf)
    H_out = (H + 2*pH - kH) // sH + 1
    W_out = (W + 2*pW - kW) // sW + 1
    out = np.zeros((C, H_out, W_out))
    for i in range(H_out):
        for j in range(W_out):
            region = x_pad[:, i*sH:i*sH+kH, j*sW:j*sW+kW]
            out[:, i, j] = region.max(axis=(1, 2))
    return out

np.random.seed(0)
x = np.random.randn(1, 4, 4)
print('输入 (1,4,4):')
print(x[0])

out = max_pool2d(x, kernel_size=2, stride=2, padding=0)
print('\nMax Pool (kernel=2, stride=2, padding=0) 输出:')
print(out[0])

x_t = torch.tensor(x, dtype=torch.float32).unsqueeze(0)
ref = F.max_pool2d(x_t, kernel_size=2, stride=2, padding=0).squeeze(0).numpy()
print('\nPyTorch 参考结果:')
print(ref[0])
print('结果一致:', np.allclose(out, ref))


输入 (1,4,4):
[[ 1.76405235  0.40015721  0.97873798  2.2408932 ]
 [ 1.86755799 -0.97727788  0.95008842 -0.15135721]
 [-0.10321885  0.4105985   0.14404357  1.45427351]
 [ 0.76103773  0.12167502  0.44386323  0.33367433]]

Max Pool (kernel=2, stride=2, padding=0) 输出:
[[1.86755799 2.2408932 ]
 [0.76103773 1.45427351]]

PyTorch 参考结果:
[[1.867558  2.2408931]
 [0.7610377 1.4542735]]
结果一致: True


### 3.1

输入输出通道数均为 $C$。

**1. 单个 $5 \times 5$ 卷积层参数量（不带偏置）**

$$C \times C \times 5 \times 5 = 25C^2$$

**2. 两个串联 $3 \times 3$ 卷积层总参数量（不带偏置）**

$$9C^2 + 9C^2 = 18C^2$$

两个 $3\times3$ 卷积（$18C^2$）比一个 $5\times5$ 卷积（$25C^2$）参数更少，感受野相同。

In [2]:
# 3.2
import torch
import torch.nn as nn

def nin_block(in_channels, out_channels, kernel_size, stride, padding):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding), nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, 1), nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, 1), nn.ReLU()
    )

block = nin_block(3, 16, kernel_size=5, stride=1, padding=2)
print(block)
x = torch.randn(1, 3, 32, 32)
print('输入形状:', x.shape)
print('输出形状:', block(x).shape)


Sequential(
  (0): Conv2d(3, 16, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (1): ReLU()
  (2): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1))
  (3): ReLU()
  (4): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1))
  (5): ReLU()
)
输入形状: torch.Size([1, 3, 32, 32])
输出形状: torch.Size([1, 16, 32, 32])


### 4.1

给定 $x_1=2, x_2=4, x_3=6, x_4=8$，$\gamma=2$，$\beta=1$，$\epsilon=0$。

**均值：** $\mu = 5$

**方差：** $\sigma^2 = [(9+1+1+9)/4] = 5$

**输出 $y_i = 2 \cdot \frac{x_i - 5}{\sqrt{5}} + 1$：**

$$y_1 = -\frac{6}{\sqrt{5}} + 1 \approx -1.683$$
$$y_2 = -\frac{2}{\sqrt{5}} + 1 \approx 0.106$$
$$y_3 = \frac{2}{\sqrt{5}} + 1 \approx 1.894$$
$$y_4 = \frac{6}{\sqrt{5}} + 1 \approx 3.683$$

In [3]:
# 4.2
import torch
import torch.nn as nn

class Residual(nn.Module):
    def __init__(self, num_channels, use_1x1conv=False, strides=1):
        super().__init__()
        self.conv1 = nn.Conv2d(num_channels, num_channels, kernel_size=3, padding=1, stride=strides)
        self.conv2 = nn.Conv2d(num_channels, num_channels, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(num_channels, num_channels, kernel_size=1, stride=strides) if use_1x1conv else None
        self.bn1 = nn.BatchNorm2d(num_channels)
        self.bn2 = nn.BatchNorm2d(num_channels)

    def forward(self, x):
        y = torch.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        if self.conv3:
            x = self.conv3(x)
        return torch.relu(y + x)

blk = Residual(16)
x = torch.randn(2, 16, 8, 8)
print('输入形状:', x.shape)
print('输出形状 (use_1x1conv=False):', blk(x).shape)

blk2 = Residual(16, use_1x1conv=True, strides=2)
print('输出形状 (use_1x1conv=True, strides=2):', blk2(x).shape)


输入形状: torch.Size([2, 16, 8, 8])
输出形状 (use_1x1conv=False): torch.Size([2, 16, 8, 8])
输出形状 (use_1x1conv=True, strides=2): torch.Size([2, 16, 4, 4])


### 5.1

**1. 底层使用小学习率的原因**

预训练网络底层已学到通用特征（边缘、纹理等），对目标任务同样适用。若用大学习率更新这些层会破坏已有的有效表示（灾难性遗忘）。顶层是随机初始化的，需从头学习目标任务的分类能力，因此设置较大学习率加速收敛。

**2. 目标数据集小且与源数据集相似时的微调策略**

冻结预训练模型的底层特征提取层（不更新参数），只训练新加入的顶层输出层。同时配合较小学习率、weight decay、dropout 等正则化手段以及数据增广来防止过拟合，训练轮数也不宜过多。

In [4]:
# 5.2
from torchvision import transforms

train_augs = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    transforms.ToTensor()
])

print(train_augs)


Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.5, 1.5), contrast=(0.5, 1.5), saturation=(0.5, 1.5), hue=None)
    ToTensor()
)


### 6.1

真实框 $A=[10,10,50,50]$，预测框 $B=[30,30,70,70]$。

**交集区域：**
$x_1=\max(10,30)=30,\ y_1=30,\ x_2=\min(50,70)=50,\ y_2=50$

$$S_{\cap} = (50-30)\times(50-30) = 400$$

**各自面积：** $S_A = 40\times40 = 1600,\ S_B = 40\times40 = 1600$

**并集：** $S_{\cup} = 1600 + 1600 - 400 = 2800$

**IoU：**
$$\text{IoU} = \frac{400}{2800} = \frac{1}{7} \approx 0.1429$$

In [5]:
# 6.2
import torch
import torch.nn.functional as F

def label_smoothing_loss(logits, targets, epsilon=0.1):
    N, K = logits.shape
    log_probs = F.log_softmax(logits, dim=1)
    smooth_labels = torch.full((N, K), epsilon / (K - 1))
    smooth_labels.scatter_(1, targets.unsqueeze(1), 1.0 - epsilon)
    loss = -(smooth_labels * log_probs).sum(dim=1).mean()
    return loss

torch.manual_seed(42)
logits = torch.randn(4, 5)
targets = torch.tensor([0, 2, 1, 4])
print('标签平滑交叉熵损失:', label_smoothing_loss(logits, targets, epsilon=0.1).item())
print('标准交叉熵损失:', F.cross_entropy(logits, targets).item())


标签平滑交叉熵损失: 1.4089381694793701
标准交叉熵损失: 1.3351008892059326
